In [1]:
! pip install openai


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint="https://ciaiciath2-foundry-dev.cognitiveservices.azure.com/",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

In [3]:
models = ["gpt-5.6-luna", "gpt-5.4-mini"]

In [4]:
try:
    response = (
        client.chat.completions.create(
            model=models[0],
            messages=[
                {
                    "role": "user",
                    "content":
                        "Say hello"
                }
            ],
        )
    )

    print(
        "SUCCESS:",
        response.choices[0]
        .message.content
    )

    print(
        "MODEL:",
        response.model
    )

except Exception as e:
    print(
        "ERROR:",
        repr(e)
    )

SUCCESS: Hello! How can I help you today?
MODEL: gpt-5.6-luna-2026-07-09


# Final Approach: RealData YAML to Graph

## Core Flow

`RealData YAML files -> yaml.safe_load -> Python graph mapper -> Apache AGE -> NL-to-Cypher`

Use one graph:

```python
GRAPH_NAME = "realdata_knowledge_spine"
```

## Who Extracts Nodes and Edges

Nodes and relationships are extracted by Python code, not by AI. The code reads known YAML keys and maps them into graph labels and relationship types.

AI is used only after the graph is built:

- `gpt-5.6-luna` for NL-to-Cypher
- `gpt-5.4-mini` for optional schema review or fallback

## Main Node Types

- `Artifact`
- `Concept`
- `Dataset`
- `Field`
- `DQRule`
- `Step`
- `Slot`
- `Playbook`

## Main Relationships

- `Artifact -> Concept`: `HAS_CONCEPT`
- `Artifact -> Dataset`: `DEFINES_DATASET`
- `Dataset -> Field`: `HAS_FIELD`
- `Dataset -> Dataset`: `RELATES_TO`
- `Artifact -> Step`: `HAS_STEP`
- `Artifact -> Slot`: `HAS_SLOT`
- `Step -> Step`: `NEXT_STEP`
- `Step -> Slot`: `USES_SLOT`
- `Concept -> Concept`: `ADJACENT_TO`
- `Artifact -> Artifact`: `DEPENDS_ON`, `PROVIDES_TO`

## Dynamic Update Approach

Create one refresh function that can be run manually from a notebook cell or automatically from a scheduler/API endpoint.

```python
refresh_graph(
    graph_name="realdata_knowledge_spine",
    source_dir="RealData",
    mode="upsert",
)
```

Supported modes:

- `upsert`: add new nodes and update existing nodes using `MERGE`
- `rebuild`: drop and recreate the graph for development/schema changes
- `sync`: upsert latest data and remove stale graph records after validation

Start with manual single-click `upsert`. Add `rebuild` for development. Add `sync` later after validation is trusted.

## Future Structure Changes

If YAML structure changes, the transformer should not guess with AI. It should:

1. Map known sections normally.
2. Store unknown sections as raw properties or `UnknownSection` review records.
3. Log unresolved references and schema drift.
4. Require a small code mapping update only when the new section needs first-class graph nodes or relationships.

This keeps graph creation deterministic while still allowing the YAML format to evolve.

## Query Safety

NL-to-Cypher must generate read-only Cypher only. Reject write keywords such as `CREATE`, `MERGE`, `DELETE`, `SET`, `REMOVE`, `DROP`, and `LOAD CSV` before execution.


# RealData Transformation and Upload

This section builds and uploads the RealData knowledge graph into Apache AGE. It follows the same database/AGE upload pattern from `tranformation.ipynb`, but the YAML parsing and graph mapping are deterministic Python code.


In [ ]:
# Install only if your notebook environment does not already have these packages.
# ! pip install pyyaml "psycopg[binary]" python-dotenv


In [1]:
import json
import os
import re
from pathlib import Path

import psycopg
import yaml
from dotenv import load_dotenv

load_dotenv(override=True)

SOURCE_DIR = Path("RealData")
GRAPH_NAME = "realdata_knowledge_spine"

PG_HOST = os.getenv("PG_HOST")
PG_PORT = os.getenv("PG_PORT")
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")


In [ ]:
def validate_age_name(name):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Invalid AGE identifier: {name}")
    return name


def cypher_value(value):
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, list):
        return "[" + ", ".join(cypher_value(item) for item in value) + "]"
    if isinstance(value, dict):
        value = json.dumps(value, sort_keys=True, default=str)

    text = str(value).replace("\\", "\\\\").replace("'", "\\'")
    return f"'{text}'"


def clean_properties(properties):
    cleaned = {}
    for key, value in properties.items():
        if value is None:
            continue
        if isinstance(value, (str, int, float, bool, list)):
            cleaned[key] = value
        elif isinstance(value, dict):
            cleaned[key] = json.dumps(value, sort_keys=True, default=str)
        else:
            cleaned[key] = str(value)
    return cleaned

In [3]:
def load_yaml_documents(source_dir=SOURCE_DIR):
    documents = []
    for path in sorted(source_dir.rglob("*.yaml")):
        with path.open("r", encoding="utf-8") as handle:
            data = yaml.safe_load(handle)
        if isinstance(data, dict):
            documents.append({"path": path, "data": data})
    return documents


yaml_documents = load_yaml_documents()
print("YAML files loaded:", len(yaml_documents))


YAML files loaded: 32


# Helper Functions

In [4]:
def artifact_id_for(path, data):
    artifact = data.get("artifact")
    if isinstance(artifact, dict) and artifact.get("id"):
        return str(artifact["id"])
    if data.get("artifact_id"):
        return str(data["artifact_id"])
    return "artifact_" + re.sub(r"[^A-Za-z0-9_]+", "_", str(path.with_suffix(""))).strip("_")


def artifact_type_for(data):
    artifact = data.get("artifact")
    if isinstance(artifact, dict) and artifact.get("type"):
        return artifact.get("type")
    return data.get("artifact_type")


SEARCH_TEXT_EXCLUDE_KEYS = {
    "search_text",
    "source_file",
}


def flatten_search_values(value):
    if value is None:
        return []
    if isinstance(value, (str, int, float, bool)):
        text = str(value).strip()
        return [text] if text else []
    if isinstance(value, list):
        values = []
        for item in value:
            values.extend(flatten_search_values(item))
        return values
    if isinstance(value, dict):
        values = []
        for key, item in value.items():
            values.append(str(key))
            values.extend(flatten_search_values(item))
        return values
    return [str(value)]


def build_search_text(properties):
    values = []
    for key, value in properties.items():
        if key in SEARCH_TEXT_EXCLUDE_KEYS:
            continue
        values.extend(flatten_search_values(value))

    normalized = []
    seen = set()
    for value in values:
        text = re.sub(r"\s+", " ", str(value).lower()).strip()
        if not text or text in seen:
            continue
        normalized.append(text)
        seen.add(text)

    # Keep a bounded property size while still capturing enough context for fallback search.
    return " | ".join(normalized)[:16000]


def add_node(nodes, label, node_id, properties):
    if not node_id:
        raise ValueError(f"Missing node id for {label}")
    key = (label, str(node_id))
    cleaned = clean_properties(properties)
    merged = {"node_id": str(node_id), **cleaned}
    merged["search_text"] = build_search_text(merged)
    if key in nodes:
        nodes[key].update(merged)
        nodes[key]["search_text"] = build_search_text(nodes[key])
    else:
        nodes[key] = merged


def add_edge(edges, source_label, source_id, rel_type, target_label, target_id, properties=None):
    if not source_id or not target_id:
        return
    edges.append({
        "source_label": source_label,
        "source_id": str(source_id),
        "rel_type": validate_age_name(rel_type),
        "target_label": target_label,
        "target_id": str(target_id),
        "properties": clean_properties(properties or {}),
    })


# Graph Edge Types and Their Purpose

The `build_graph_records()` function creates **15 directional edge types** in two passes:

1. **First pass:** creates nodes and parent-child relationships.
2. **Second pass:** creates cross-references after all nodes have been discovered.

---

## 1. Parent-Child Edges

These edges describe ownership or containment.

| Edge | Direction | Purpose |
|---|---|---|
| `HAS_CONCEPT` | `Artifact → Concept` | The artifact contains or defines a business concept. |
| `DEFINES_DATASET` | `Artifact → Dataset` | The artifact defines a dataset. |
| `HAS_FIELD` | `Dataset → Field` | The dataset contains a canonical field. |
| `HAS_DQ_STANDARD` | `Dataset → DQRule` | The dataset has a data-quality requirement. |
| `HAS_STEP` | `Artifact → Step` | The artifact contains a workflow step. |
| `HAS_SLOT` | `Artifact → Slot` | The artifact contains a reusable input or information slot. |
| `HAS_PLAYBOOK` | `Artifact → Playbook` | The artifact contains or references a playbook. |

Example:

```text
(Artifact)-[:DEFINES_DATASET]->(Dataset)-[:HAS_FIELD]->(Field)
```

This can answer: *Which datasets are defined by an artifact, and which fields belong to those datasets?*

---

## 2. Cross-Reference Edges

These edges connect nodes that may be defined in different YAML documents.

| Edge | Direction | Purpose |
|---|---|---|
| `DEPENDS_ON` | `Artifact → Artifact` | One artifact requires another artifact. |
| `PROVIDES_TO` | `Artifact → Artifact` | One artifact supplies information or output to another. |
| `ADJACENT_TO` | `Concept → Concept` | Two business concepts are semantically related. |
| `RELATES_TO` | `Dataset → Dataset` | Two datasets have a join or business relationship. |
| `NEXT_STEP` | `Step → Step` | Defines workflow execution order. |
| `USES_SLOT` | `Step → Slot` | A workflow step consumes an input slot. |
| `REFERENCES_CONCEPT` | `Step → Concept` | A workflow step refers to a business concept. |
| `USED_BY_STEP` | `Slot → Step` | Reverse navigation from a slot to a step using it. |

---

## 3. Artifact Dependencies

```text
(Artifact)-[:DEPENDS_ON]->(Artifact)
```

The source artifact requires the target artifact.

```text
(MMM Use Case)-[:DEPENDS_ON]->(Marketing Foundation)
```

`PROVIDES_TO` describes the information flow in the other direction:

```text
(Artifact)-[:PROVIDES_TO]->(Artifact)
```

---

## 4. Dataset Relationships

```text
(Dataset)-[:RELATES_TO]->(Dataset)
```

A `RELATES_TO` edge can retain the complete YAML relationship object, including join keys, cardinality, type, and description.

```text
(ClaimsDataset)-[:RELATES_TO {
    relationship_type: "many_to_one",
    join_key: "payer_id"
}]->(PayerDataset)
```

The edge is created with:

```python
add_edge(
    edges,
    "Dataset",
    entity_id,
    "RELATES_TO",
    "Dataset",
    target_entity,
    rel,
)
```

---

## 5. Workflow Order

```text
(Step)-[:NEXT_STEP]->(Step)
```

This defines which workflow step follows another:

```text
(LoadData)-[:NEXT_STEP]->(ValidateData)
```

---

## 6. Step and Slot Relationships

```text
(Step)-[:USES_SLOT]->(Slot)
(Slot)-[:USED_BY_STEP]->(Step)
```

Both edges represent the same association in opposite directions. Because graph databases can traverse an edge backward, `USED_BY_STEP` is usually redundant.

The reverse lookup can use only `USES_SLOT`:

```cypher
MATCH (slot:Slot)<-[:USES_SLOT]-(step:Step)
RETURN slot, step
```

---

## 7. Step and Concept Relationships

```text
(Step)-[:REFERENCES_CONCEPT]->(Concept)
```

This connects procedural knowledge to business terminology.

```text
(CalculateROI)-[:REFERENCES_CONCEPT]->(ReturnOnInvestment)
```

---

## 8. Concept Adjacency

```text
(Concept)-[:ADJACENT_TO]->(Concept)
```

This represents a semantic relationship between concepts:

```text
(ROI)-[:ADJACENT_TO]->(MarketingInvestment)
```

The code creates this edge only when the target exists in `concept_ids`.

---

## 9. Why Two Passes Are Used

### First pass: ownership structure

```text
Artifact → Concept
Artifact → Dataset → Field
Artifact → Dataset → DQRule
Artifact → Step
Artifact → Slot
Artifact → Playbook
```

### Second pass: cross-references

```text
Artifact → Artifact
Concept → Concept
Dataset → Dataset
Step → Step
Step → Slot
Step → Concept
```

A referenced node may be declared in a later YAML document, so cross-reference edges are created only after the first pass discovers the nodes.

---

## 10. Validation Limitation

The function collects these identifier sets:

```python
concept_ids = set()
dataset_ids = set()
step_ids = set()
slot_ids = set()
```

Currently, only `concept_ids` is used to validate a relationship target:

```python
if str(adjacent_id) in concept_ids:
    add_edge(...)
```

The following edges can therefore reference an unknown target unless additional validation is added:

- `DEPENDS_ON`
- `PROVIDES_TO`
- `RELATES_TO`
- `NEXT_STEP`
- `USES_SLOT`
- `REFERENCES_CONCEPT`
- `USED_BY_STEP`

The collected ID sets should be used to validate every cross-reference before adding its edge.

---

## Summary

| Category | Relationships |
|---|---|
| Ownership | `HAS_CONCEPT`, `DEFINES_DATASET`, `HAS_FIELD`, `HAS_DQ_STANDARD`, `HAS_STEP`, `HAS_SLOT`, `HAS_PLAYBOOK` |
| Data and knowledge | `DEPENDS_ON`, `PROVIDES_TO`, `ADJACENT_TO`, `RELATES_TO`, `REFERENCES_CONCEPT` |
| Workflow | `NEXT_STEP`, `USES_SLOT`, `USED_BY_STEP` |

Together, these relationships represent metadata, datasets, business concepts, workflows, dependencies, playbooks, and data-quality rules.

# Graph Node Types and How Nodes Are Defined

The `build_graph_records()` function defines **8 node labels**. These nodes represent structured knowledge and metadata from the YAML documents; they are not raw transactional rows from a source database.

| Node label | YAML source | Node ID | Purpose |
|---|---|---|---|
| `Artifact` | One complete YAML document | `artifact.id`, then `artifact_id`, otherwise a path-derived ID | Represents the top-level knowledge artifact. |
| `Concept` | Each item in `concepts` | `concept_id` | Represents a business term, definition, or semantic concept. |
| `Dataset` | Each item in `datasets` | `entity_id` | Represents a logical dataset or business entity. |
| `Field` | Each item in `dataset.fields` | `<entity_id>.<canonical_name>` | Represents a canonical field belonging to a dataset. |
| `DQRule` | Each item in `dataset.dq_standards` | `dq_rule_id`, otherwise a derived dataset/field/rule ID | Represents a data-quality requirement. |
| `Step` | Each item in `steps` | `step_id` | Represents a workflow or process step. |
| `Slot` | Each item in `slots` | `slot_id` | Represents an input, output, or reusable information placeholder. |
| `Playbook` | Items or entries in `playbooks` | `playbook_id`, `id`, mapping key/index, or scalar value | Represents an operational or analytical playbook. |

---

## 1. Common Node Construction

Every node is created through:  

```python
add_node(nodes, label, node_id, properties)
```

The helper performs the following operations:

1. Rejects a missing node ID.
2. Creates a unique dictionary key using `(label, node_id)`.
3. Cleans unsupported or null properties.
4. Adds a common `node_id` property.
5. Builds normalized `search_text` for fallback retrieval.
6. Merges properties when the same label and ID are encountered again.

```python
key = (label, str(node_id))
merged = {"node_id": str(node_id), **cleaned}
merged["search_text"] = build_search_text(merged)
```

Because the label is part of the dictionary key, `("Concept", "roi")` and `("Dataset", "roi")` are treated as different nodes.

---

## 2. Artifact Nodes

An `Artifact` is created once for each YAML document. Its ID is resolved in this order:

```text
artifact.id → artifact_id → ID derived from the source path
```

Important properties include `artifact_id`, `artifact_type`, `title`, schema/content versions, `layer`, `scope`, `use_case_id`, `subject_area`, review metadata, and `source_file`.

---

## 3. Concept Nodes

Each mapping in `concepts` becomes a `Concept` node when it contains `concept_id`. All YAML properties are retained, and the function adds `artifact_id` and `source_file`.

```text
Concept ID = concept_id
```

---

## 4. Dataset, Field, and DQRule Nodes

A `Dataset` uses `entity_id` as its node ID. Nested `fields`, `relationships`, and `dq_standards` are excluded from dataset properties because they are represented separately.

A `Field` uses a composite ID so the same field name can safely exist in different datasets:

```python
field_id = f"{entity_id}.{canonical_name}"
```

A `DQRule` uses its explicit ID when available. Otherwise, the function derives one:

```python
dq_id = dq.get("dq_rule_id") or (
    f"{entity_id}.{dq.get('field', 'dataset')}.{dq.get('rule_ref', 'rule')}"
)
```

---

## 5. Step and Slot Nodes

A `Step` requires `step_id`, and a `Slot` requires `slot_id`. Their complete YAML mappings are retained along with `artifact_id` and `source_file`. Relationships such as `NEXT_STEP`, `USES_SLOT`, and `REFERENCES_CONCEPT` are created separately as edges.

---

## 6. Playbook Nodes

`playbooks` may be a list or a dictionary. A playbook ID is selected from `playbook_id`, `id`, its dictionary key/list index, or its scalar value. Scalar entries are normalized into properties such as `name` and `raw_value`.

---

## 7. Common Properties

Most child nodes receive these traceability properties:

| Property | Purpose |
|---|---|
| `node_id` | Common unique identifier used during graph upload and matching. |
| `artifact_id` | Identifies the parent YAML artifact. |
| `source_file` | Records which YAML file produced the node. |
| `search_text` | Lowercase flattened text used by fallback keyword retrieval. |

`search_text` is generated from scalar, list, and dictionary properties, deduplicated, normalized, and limited to 16,000 characters. `source_file` is intentionally excluded from it.

---

## 8. Counting Node Instances by Type

The function defines **8 node types**, but the number of node instances depends on the YAML content. Count instances per label with:

```python
from collections import Counter

node_counts_by_type = Counter(label for label, node_id in nodes)
print(dict(sorted(node_counts_by_type.items())))
```

The existing `len(nodes)` output reports the total number of unique node instances across all eight labels.

# Graph Construction: Team Summary

## Node Types

| Node | Represents | ID source |
|---|---|---|
| `Artifact` | Complete YAML knowledge artifact | Artifact ID or source path |
| `Concept` | Business term or definition | `concept_id` |
| `Dataset` | Logical dataset or entity | `entity_id` |
| `Field` | Dataset field | `entity_id.canonical_name` |
| `DQRule` | Data-quality requirement | `dq_rule_id` or derived ID |
| `Step` | Workflow step | `step_id` |
| `Slot` | Workflow input or output | `slot_id` |
| `Playbook` | Operational playbook | Playbook ID, key, index, or value |

## Edge Types

| Edge | From → To | Meaning |
|---|---|---|
| `HAS_CONCEPT` | Artifact → Concept | Artifact defines a concept |
| `DEFINES_DATASET` | Artifact → Dataset | Artifact defines a dataset |
| `HAS_FIELD` | Dataset → Field | Dataset contains a field |
| `HAS_DQ_STANDARD` | Dataset → DQRule | Dataset has a quality rule |
| `HAS_STEP` | Artifact → Step | Artifact contains a workflow step |
| `HAS_SLOT` | Artifact → Slot | Artifact contains a slot |
| `HAS_PLAYBOOK` | Artifact → Playbook | Artifact contains a playbook |
| `DEPENDS_ON` | Artifact → Artifact | Artifact depends on another artifact |
| `PROVIDES_TO` | Artifact → Artifact | Artifact provides output to another artifact |
| `ADJACENT_TO` | Concept → Concept | Concepts are related |
| `RELATES_TO` | Dataset → Dataset | Datasets have a join or business relationship |
| `NEXT_STEP` | Step → Step | Defines workflow order |
| `USES_SLOT` | Step → Slot | Step uses a slot |
| `REFERENCES_CONCEPT` | Step → Concept | Step refers to a business concept |
| `USED_BY_STEP` | Slot → Step | Reverse navigation from slot to step |

## Construction Flow

| Phase | Operation | Purpose |
|---|---|---|
| 1. Load | Read all YAML documents | Collect structured knowledge |
| 2. Create nodes | Build the eight node types | Represent YAML objects in the graph |
| 3. Parent edges | Connect artifacts to contained objects | Build the ownership hierarchy |
| 4. Cross-reference edges | Connect datasets, concepts, steps, and artifacts | Build relationships across YAML files |
| 5. Normalize | Clean properties and create `search_text` | Support consistent storage and retrieval |
| 6. Deduplicate | Use `(label, node_id)` as the key | Prevent duplicate nodes |
| 7. Upload | Write nodes and edges to PostgreSQL AGE | Persist the graph |
| 8. Verify | Count nodes and edges and inspect references | Confirm successful construction |

## One-Line Architecture

| Input | Transformation | Output |
|---|---|---|
| YAML metadata and structured knowledge | Convert YAML objects into typed nodes and references into directional edges | One connected PostgreSQL AGE knowledge graph |

# Graph Creation Function

In [ ]:
def build_graph_records(yaml_documents):
    nodes = {}
    edges = []
    artifact_aliases = {}
    concept_ids = set()
    dataset_ids = set()
    step_ids = set()
    slot_ids = set()

    # First pass: create nodes and parent-child edges.
    for doc in yaml_documents:
        path = doc["path"]
        data = doc["data"]
        source_file = str(path)
        artifact_id = artifact_id_for(path, data)
        artifact_aliases[artifact_id] = artifact_id

        artifact = data.get("artifact") if isinstance(data.get("artifact"), dict) else {}
        artifact_props = {
            "artifact_id": artifact_id,
            "artifact_type": artifact_type_for(data),
            "title": data.get("title") or artifact.get("title"),
            "schema_version": data.get("schema_version") or artifact.get("schema_version"),
            "content_version": data.get("content_version") or artifact.get("content_version"),
            "layer": data.get("layer"),
            "scope": data.get("scope"),
            "use_case_id": data.get("use_case_id"),
            "subject_area": data.get("subject_area"),
            "review_status": data.get("review_status"),
            "last_reviewed": data.get("last_reviewed"),
            "source_file": source_file,
        }
        add_node(nodes, "Artifact", artifact_id, artifact_props)

        for concept in data.get("concepts") or []:
            concept_id = concept.get("concept_id")
            if not concept_id:
                continue
            concept_ids.add(str(concept_id))
            add_node(nodes, "Concept", concept_id, {
                **concept,
                "concept_id": concept_id,
                "artifact_id": artifact_id,
                "source_file": source_file,
            })
            add_edge(edges, "Artifact", artifact_id, "HAS_CONCEPT", "Concept", concept_id)

        for dataset in data.get("datasets") or []:
            entity_id = dataset.get("entity_id")
            if not entity_id:
                continue
            dataset_ids.add(str(entity_id))
            dataset_props = {k: v for k, v in dataset.items() if k not in {"fields", "relationships", "dq_standards"}}
            add_node(nodes, "Dataset", entity_id, {
                **dataset_props,
                "entity_id": entity_id,
                "artifact_id": artifact_id,
                "source_file": source_file,
            })
            add_edge(edges, "Artifact", artifact_id, "DEFINES_DATASET", "Dataset", entity_id)

            for field in dataset.get("fields") or []:
                canonical_name = field.get("canonical_name")
                if not canonical_name:
                    continue
                field_id = f"{entity_id}.{canonical_name}"
                add_node(nodes, "Field", field_id, {
                    **field,
                    "field_id": field_id,
                    "entity_id": entity_id,
                    "canonical_name": canonical_name,
                    "artifact_id": artifact_id,
                    "source_file": source_file,
                })
                add_edge(edges, "Dataset", entity_id, "HAS_FIELD", "Field", field_id)

            for dq in dataset.get("dq_standards") or []:
                dq_id = dq.get("dq_rule_id") or f"{entity_id}.{dq.get('field', 'dataset')}.{dq.get('rule_ref', 'rule')}"
                add_node(nodes, "DQRule", dq_id, {
                    **dq,
                    "dq_rule_id": dq_id,
                    "entity_id": entity_id,
                    "artifact_id": artifact_id,
                    "source_file": source_file,
                })
                add_edge(edges, "Dataset", entity_id, "HAS_DQ_STANDARD", "DQRule", dq_id)

        for step in data.get("steps") or []:
            step_id = step.get("step_id")
            if not step_id:
                continue
            step_ids.add(str(step_id))
            add_node(nodes, "Step", step_id, {
                **step,
                "step_id": step_id,
                "artifact_id": artifact_id,
                "source_file": source_file,
            })
            add_edge(edges, "Artifact", artifact_id, "HAS_STEP", "Step", step_id)

        for slot in data.get("slots") or []:
            slot_id = slot.get("slot_id")
            if not slot_id:
                continue
            slot_ids.add(str(slot_id))
            add_node(nodes, "Slot", slot_id, {
                **slot,
                "slot_id": slot_id,
                "artifact_id": artifact_id,
                "source_file": source_file,
            })
            add_edge(edges, "Artifact", artifact_id, "HAS_SLOT", "Slot", slot_id)

        playbooks = data.get("playbooks") or []
        if isinstance(playbooks, dict):
            playbook_items = list(playbooks.items())
        else:
            playbook_items = list(enumerate(playbooks))

        for index_or_key, playbook in playbook_items:
            if isinstance(playbook, dict):
                playbook_props = playbook
                playbook_id = playbook.get("playbook_id") or playbook.get("id") or str(index_or_key)
            else:
                playbook_id = str(playbook)
                playbook_props = {"name": str(playbook), "raw_value": str(playbook)}

            if not playbook_id:
                playbook_id = f"{artifact_id}.playbook.{index_or_key}"

            add_node(nodes, "Playbook", playbook_id, {
                **playbook_props,
                "playbook_id": playbook_id,
                "artifact_id": artifact_id,
                "source_file": source_file,
            })
            add_edge(edges, "Artifact", artifact_id, "HAS_PLAYBOOK", "Playbook", playbook_id)

    # Second pass: create cross-reference edges after all nodes exist.
    for doc in yaml_documents:
        path = doc["path"]
        data = doc["data"]
        artifact_id = artifact_id_for(path, data)

        for dependency in data.get("depends_on") or []:
            add_edge(edges, "Artifact", artifact_id, "DEPENDS_ON", "Artifact", dependency)
        for target in data.get("provides_to") or []:
            add_edge(edges, "Artifact", artifact_id, "PROVIDES_TO", "Artifact", target)

        for concept in data.get("concepts") or []:
            concept_id = concept.get("concept_id")
            for adjacent_id in concept.get("adjacent_concepts") or []:
                if str(adjacent_id) in concept_ids:
                    add_edge(edges, "Concept", concept_id, "ADJACENT_TO", "Concept", adjacent_id)

        for dataset in data.get("datasets") or []:
            entity_id = dataset.get("entity_id")
            for rel in dataset.get("relationships") or []:
                target_entity = rel.get("to_entity")
                if target_entity:
                    add_edge(edges, "Dataset", entity_id, "RELATES_TO", "Dataset", target_entity, rel)

        for step in data.get("steps") or []:
            step_id = step.get("step_id")
            if step.get("next_step"):
                add_edge(edges, "Step", step_id, "NEXT_STEP", "Step", step["next_step"])
            for slot_id in step.get("uses_slots") or []:
                add_edge(edges, "Step", step_id, "USES_SLOT", "Slot", slot_id)
            for concept_id in step.get("concept_refs") or []:
                add_edge(edges, "Step", step_id, "REFERENCES_CONCEPT", "Concept", concept_id)

        for slot in data.get("slots") or []:
            slot_id = slot.get("slot_id")
            for step_id in slot.get("used_by_steps") or []:
                add_edge(edges, "Slot", slot_id, "USED_BY_STEP", "Step", step_id)

    return nodes, edges


nodes, edges = build_graph_records(yaml_documents)
print("Nodes:", len(nodes))
print("Edges:", len(edges))


In [6]:
def connect_postgres():
    return psycopg.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DATABASE,
        user=PG_USER,
        password=PG_PASSWORD,
    )


def init_age(cursor):
    cursor.execute("LOAD 'age';")
    cursor.execute('SET search_path = ag_catalog, "$user", public;')


In [ ]:
def ensure_graph(conn, graph_name=GRAPH_NAME, rebuild=False):
    graph_name = validate_age_name(graph_name)
    with conn.cursor() as cursor:
        init_age(cursor)
        cursor.execute("SELECT 1 FROM ag_catalog.ag_graph WHERE name = %s;", (graph_name,))
        exists = cursor.fetchone() is not None
        if exists and rebuild:
            cursor.execute("SELECT drop_graph(%s, true);", (graph_name,))
            exists = False
        if not exists:
            cursor.execute("SELECT create_graph(%s);", (graph_name,))
    conn.commit()


def merge_node(cursor, graph_name, label, properties):
    label = validate_age_name(label)
    node_id = properties["node_id"]
    props = ", ".join(f"{key}: {cypher_value(value)}" for key, value in properties.items())
    query = f"""
    SELECT *
    FROM cypher('{graph_name}', $$
        MERGE (n:{label} {{node_id: {cypher_value(node_id)}}})
        SET n += {{{props}}}
        RETURN n
    $$) AS (n agtype);
    """
    cursor.execute(query)
    cursor.fetchone()


def merge_edge(cursor, graph_name, edge):
    rel_type = validate_age_name(edge["rel_type"])
    props = edge.get("properties") or {}
    prop_text = ""
    if props:
        prop_text = " SET r += {" + ", ".join(f"{key}: {cypher_value(value)}" for key, value in props.items()) + "}"
    query = f"""
    SELECT *
    FROM cypher('{graph_name}', $$
        MATCH (a {{node_id: {cypher_value(edge['source_id'])}}})
        MATCH (b {{node_id: {cypher_value(edge['target_id'])}}})
        MERGE (a)-[r:{rel_type}]->(b)
        {prop_text}
        RETURN r
    $$) AS (r agtype);
    """
    cursor.execute(query)
    cursor.fetchone()


In [ ]:
def refresh_graph(mode="upsert", graph_name=GRAPH_NAME, source_dir=SOURCE_DIR):
    if mode not in {"upsert", "rebuild"}:
        raise ValueError("This notebook supports mode='upsert' or mode='rebuild'. Add stale-delete validation before sync mode.")

    documents = load_yaml_documents(source_dir)
    nodes, edges = build_graph_records(documents)

    conn = connect_postgres()
    try:
        ensure_graph(conn, graph_name=graph_name, rebuild=(mode == "rebuild"))
        with conn.cursor() as cursor:
            init_age(cursor)
            for (label, _node_id), properties in nodes.items():
                merge_node(cursor, graph_name, label, properties)
            for edge in edges:
                merge_edge(cursor, graph_name, edge)
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

    return {
        "mode": mode,
        "graph_name": graph_name,
        "yaml_files": len(documents),
        "nodes": len(nodes),
        "edges": len(edges),
    }


In [13]:
# Single-click manual refresh.
# Use mode="upsert" for normal updates and mode="rebuild" for development/schema changes.
summary = refresh_graph(mode="upsert")
print(summary)


{'mode': 'upsert', 'graph_name': 'realdata_knowledge_spine', 'yaml_files': 32, 'nodes': 3748, 'edges': 5000}


In [14]:
def verify_graph_counts(graph_name=GRAPH_NAME):
    conn = connect_postgres()
    try:
        with conn.cursor() as cursor:
            init_age(cursor)
            cursor.execute(f"""
            SELECT *
            FROM cypher('{graph_name}', $$
                MATCH (n)
                RETURN count(n)
            $$) AS (node_count agtype);
            """)
            node_count = cursor.fetchone()[0]

            cursor.execute(f"""
            SELECT *
            FROM cypher('{graph_name}', $$
                MATCH ()-[r]->()
                RETURN count(r)
            $$) AS (edge_count agtype);
            """)
            edge_count = cursor.fetchone()[0]
    finally:
        conn.close()
    return {"nodes": node_count, "edges": edge_count}


verify_graph_counts()


{'nodes': '3748', 'edges': '5026'}